# Ghana Fire RP1 Analysis v1.2.3

This canonical notebook is the scientific narrative and **thin orchestration layer** for the governed Ghana landscape-fire analysis. Scientific algorithms, statistical estimators, validation logic, publication schemas and renderer policies are implemented in `rp1_analysis_v1` and governed by the five configuration contracts. The notebook loads those authorities, executes them in the prescribed order, presents their outputs and records reproducibility evidence; it is not an independent scientific specification.


# SECTION 00 — Scaffold / authority

The scaffold establishes configuration, governed input validity, canonical release identity and analytical populations before any research question is evaluated. Runtime scientific admissibility is package-owned; canonical byte identity is recorded separately for provenance and release-integrity comparison.

### 00-01 — Package, run and configuration authority

**Scientific purpose:** Establish the package-backed execution context and load all five scientific/output contracts.

**Theoretical/methodological basis:** Analysis parameters are prospective configuration authority, not notebook-owned constants. Individual configuration hashes and their aggregate identity provide a reproducible specification boundary.

**Inputs:** Installed/local `rp1_analysis_v1` package and the repository `config/` directory discovered from package provenance.

**Method:** Resolve portable project paths, create a run identity, load the configuration bundle, and expose the five file hashes plus aggregate configuration hash.

**Expected output:** A run-identity record and five-contract hash authority.

**Interpretation limits:** Run timestamps are provenance only and do not enter scientific estimates or deterministic publication filenames.


In [ ]:
from rp1_analysis_v1.trend_robustness import PRIMARY_TREND_COLUMNS
from dataclasses import asdict
from datetime import UTC, datetime
import json
import os

import pandas as pd
from IPython.display import display

from rp1_analysis_v1 import (
    ProjectPaths,
    build_publication_run_isolated,
    build_rq1_tables,
    build_rq3_tables,
    build_secondary_input_run_isolated,
    finalise_interactive_run,
    load_configuration_bundle,
    load_data_authorities,
    validate_and_integrate_satscan_results_if_available,
    validate_canonical_integration,
    validate_data_authorities,
    validate_governed_inputs,
    validate_secondary_completion_state,
)
from rp1_analysis_v1.config import (
    export_realised_configuration,
    rq1_spatial_parameters,
    rq2_trend_parameters,
    rq3_model_parameters,
    satscan_parameters,
)
from rp1_analysis_v1.gee import configured_predictor_metadata
from rp1_analysis_v1.outputs import OutputWriter
from rp1_analysis_v1.provenance import RunProvenance, inventory_inputs
from rp1_analysis_v1.registers import artefact_registry
from rp1_analysis_v1.secondary_clusters import EXTERNAL_RESULTS_REQUIRED, RESULTS_VALIDATED
from rp1_analysis_v1.trend_robustness import build_rq2_tables

paths = ProjectPaths.discover()
contracts = load_configuration_bundle(paths.root / "config")
run_tag = os.environ.get("RP1_NOTEBOOK_RUN_ID") or datetime.now(UTC).strftime("canonical_%Y%m%dT%H%M%S_%fZ")
run_ids = {
    "secondary": f"{run_tag}_sec",
    "publication": f"{run_tag}_pub",
    "closeout": f"{run_tag}_close",
}
config_hash_authority = pd.DataFrame(
    [{"contract": name, "sha256": digest} for name, digest in contracts.file_hashes.items()]
)
config_hash_authority["aggregate_configuration_sha256"] = contracts.configuration_sha256
display(pd.DataFrame([{"run_tag": run_tag, "package_version": contracts.package_version}]))
display(config_hash_authority)

### 00-02 — Governed input validation, identity and spatial authorities

**Scientific purpose:** Establish that the current analytical inputs satisfy the complete governed scientific/data contract before analysis, while separately recording whether their bytes equal the files shipped with the canonical release.

**Theoretical/methodological basis:** Scientific admissibility depends on schema, key, support, denominator, geometry, reconciliation and population rules. SHA-256 identity remains mandatory provenance but is not itself a scientific validity criterion.

**Inputs:** `data/input_sha256.json`, current governed panel/metadata files and geometry members resolved through `ProjectPaths`.

**Method:** Invoke the package-owned governed-input validator. Require `input_validation_status == PASS`; record `canonical_input_identity` as `MATCH` or `DIFFERENT`; then load the validated authorities for downstream package builders.

**Expected output:** Scientific/data validation PASS plus canonical identity state and exact current input hashes.

**Interpretation limits:** `MATCH` means exact canonical-release bytes. `DIFFERENT` is not a failure when the current files satisfy the complete governed contract. Neither state validates a remote-sensing product against ground truth.

In [ ]:
input_validation = validate_governed_inputs(paths, bundle=contracts)
input_validation_status = input_validation["input_validation_status"]
canonical_input_identity = input_validation["canonical_input_identity"]
assert input_validation_status == "PASS"
assert canonical_input_identity in {"MATCH", "DIFFERENT"}

input_inventory = pd.DataFrame(input_validation["current_input_identities"])
authorities = load_data_authorities(paths, contracts.data_schema)
data_contract = validate_data_authorities(
    authorities,
    contracts.analysis,
    contracts.data_schema,
)

display(input_inventory)
display(pd.DataFrame([{
    "input_validation_status": input_validation_status,
    "canonical_input_identity": canonical_input_identity,
}]))
display(pd.DataFrame([
    {
        "panel": panel_id,
        "expected_rows": spec.expected_rows,
        "expected_units": spec.expected_units,
        "actual_rows": len(authorities.acz_panel if panel_id == "acz_monthly" else authorities.district_panel),
        "actual_units": (authorities.acz_panel if panel_id == "acz_monthly" else authorities.district_panel)[contracts.analysis.field_roles["keys"]["unit_id"]].nunique(),
    }
    for panel_id, spec in contracts.data_schema.panels.items()
]))

### 00-03 — Analytical-population audit and scaffold gate

**Scientific purpose:** Reproduce the governed ACZ, district, long-run, paired-overlap and RQ3 model populations before substantive analysis.

**Theoretical/methodological basis:** Eligibility follows source support and positive denominator rules fixed in configuration. A population mismatch is a stop condition, not an invitation to tune exclusions.

**Inputs:** Validated data-contract summary and configured population expectations.

**Method:** Compare realised unit/row support with the configured populations and verify district-to-ACZ reconciliation.

**Expected output:** Population audit and scaffold gate PASS.

**Interpretation limits:** Population eligibility is a design constraint; it does not imply causal representativeness beyond the observation system.


In [ ]:
population_audit = data_contract.population_authority.copy()
configured_populations = contracts.analysis.study["populations"]
assert data_contract.acz_count == int(configured_populations["acz_monthly"]["units"])
assert data_contract.district_spatial_universe_count == int(configured_populations["district_monthly"]["units"])
assert data_contract.long_run_eligible_count == int(configured_populations["long_run_district"]["units"])
assert data_contract.paired_eligible_count == int(configured_populations["paired_overlap"]["units"])
assert bool(data_contract.reconciliation["passed"].all())
scaffold_gate = "PASS"
display(population_audit)
display(data_contract.reconciliation)
display(pd.DataFrame([{"section": "00", "gate": scaffold_gate}]))

# SECTION 01 — RQ1

RQ1 asks how mapped MCD64A1 burned-area magnitude, district heterogeneity and seasonal timing depend on spatial support. ACZ and district results are complementary views of the same mapped burned-surface construct. District departures are raw differences from their parent-ACZ rate, and spatial association is evaluated with configured first-order Queen weights, permutation inference and multiplicity-controlled Local Moran statistics.


Local Moran conditional-permutation p-values form the configured eligible-district family and are controlled with Benjamini–Hochberg false-discovery-rate adjustment; inferential HH/LL/HL/LH labels are emitted only after that multiplicity control.

### 01-01 — Spatial-scale, seasonality and spatial-association authority

**Scientific purpose:** Execute the complete RQ1 analysis from the governed fixed-support population.

**Theoretical/methodological basis:** The modifiable-areal-unit problem motivates explicit comparison of ecological and administrative supports. Median/IQR and Gini summarise district long-run rates without arbitrary top-k thresholds. Circular timing respects December–January adjacency. Moran inference uses configuration-defined first-order Queen weights, one national Global Moran result, conditional Local Moran permutations and BH-FDR.

**Inputs:** Validated authorities and the RQ1 configuration section.

**Method:** Call the package RQ1 builder using immutable parameters projected from `analysis_contract.yml`; compute fixed-support rates, district heterogeneity, raw parent-ACZ departures, circular mean/resultant length, Queen weights, Global Moran, Local Moran and BH-FDR.

**Expected output:** Configuration-registered RQ1 source authorities for the integrated table, spatial/seasonal figures and supplementary district/spatial-statistics tables.

**Interpretation limits:** Spatial associations and departures are observational; they are not ignition causes, causal risk effects or automatically defined hotspots.


Local Moran conditional-permutation p-values form the configured eligible-district family and are controlled with Benjamini–Hochberg false-discovery-rate adjustment; inferential HH/LL/HL/LH labels are emitted only after that multiplicity control.

In [ ]:
rq1_parameters = rq1_spatial_parameters(contracts.analysis)
rq1 = build_rq1_tables(authorities, data_contract, contracts.analysis)
rq1_parameter_audit = pd.DataFrame([asdict(rq1_parameters)])
display(rq1_parameter_audit)
display(rq1.district_metrics_source.head())
display(rq1.spatial_statistics_authority.head())
display(pd.DataFrame([rq1.rq1_realised_run_metadata["population"]]))

### 01-02 — RQ1 acceptance gate

**Scientific purpose:** Confirm that every configured RQ1 analytical component completed on the unchanged population.

**Theoretical/methodological basis:** A gate checks execution completeness and configured-method identity; it does not add a new statistical test.

**Inputs:** RQ1 authorities and configured output registry.

**Method:** Validate non-empty manuscript/supplement sources, reconciliation and finite Global Moran authority, then record the RQ1 gate.

**Expected output:** RQ1 gate PASS.

**Interpretation limits:** PASS means the governed analysis executed as specified, not that every local association or spatial contrast is statistically supported.


In [ ]:
assert not rq1.district_metrics_source.empty
assert not rq1.spatial_statistics_authority.empty
assert bool(rq1.reconciliation["passed"].all())
assert rq1.rq1_realised_run_metadata["population"]["district_units"] == int(contracts.analysis.study["populations"]["long_run_district"]["units"])
assert rq1.spatial_statistics_authority["record_type"].astype(str).eq("global_moran_national").sum() == 1
rq1_gate = "PASS"
display(pd.DataFrame([{"section": "01", "gate": rq1_gate}]))

# SECTION 02 — RQ2

RQ2 evaluates monotonic change in the five complete fixed-support ACZ annual mapped burned-area-rate series. Sen’s slope is the effect-size estimator. The sole inferential authority is the prospectively frozen Romano–Tirlea studentized global Mann–Kendall permutation test, with the empirical-CDF convention, rule-derived bandwidth, variance floor, per-permutation re-studentisation, plus-one Monte Carlo p-value and five-test Benjamini–Hochberg family supplied by executable configuration.


### 02-01 — Production trend authority

**Scientific purpose:** Estimate monotonic direction/magnitude and serial-dependence-aware inferential support for each of the five ACZ annual fixed-support rate series.

**Theoretical/methodological basis:** Sen’s slope estimates effect magnitude independently of the studentized permutation test. The Romano–Tirlea procedure recomputes the long-run variance and studentization for every permutation; the rule-derived bandwidth is execution metadata rather than a second configuration constant.

**Inputs:** Governed ACZ monthly panel and frozen RQ2 configuration/method authorities.

**Method:** Build five complete 2001–2024 annual rate series, estimate Sen slope, execute the configured 9,999-permutation studentized global Mann–Kendall test, and apply one BH family containing exactly the five raw p-values.

**Expected output:** Five-row trend authority, 120-row annual-series authority, realised bandwidth/seed/permutation metadata and production figure/source tables.

**Interpretation limits:** Inference follows the configured Sen-slope, studentized-permutation and five-member BH authorities without outcome-driven method selection.


In [ ]:
rq2_config = contracts.analysis.research_questions["rq2"]
rq2_parameters = rq2_trend_parameters(contracts.analysis)
rq2 = build_rq2_tables(
    authorities,
    data_contract,
    contracts.analysis,
    contracts.methods,
    configuration_sha256=contracts.configuration_sha256,
)
rq2_annual_series = rq2.annual_acz_series
rq2_parameter_audit = pd.DataFrame([asdict(rq2_parameters)])
display(rq2_parameter_audit)
display(rq2.primary_trend_authority)
display(rq2.rq2_full_authority.head())
display(rq2.realised_run_metadata)


### 02-02 — RQ2 production gate

**Scientific purpose:** Verify the exact five-series/five-test production contract and realised Monte Carlo settings.

**Theoretical/methodological basis:** A valid result requires complete fixed-support annual series, one Sen effect estimate and one studentized permutation p-value per ACZ, the rule-derived bandwidth, non-zero plus-one p-values and one prespecified five-member BH family.

**Inputs:** RQ2 annual and trend authorities.

**Method:** Check series count/length, 2001–2024 coverage, fixed denominator, realised bandwidth, permutation count, recorded seed, non-zero raw p-values, BH q-values and the exact production result schema.

**Expected output:** RQ2 gate PASS.

**Interpretation limits:** The gate validates the prospectively frozen inferential path; it does not select parameters according to Ghana results.


In [ ]:
expected_rq2_rows = int(rq2_config["expected_series_count"]) * int(rq2_config["expected_n_per_series"])
assert len(rq2_annual_series) == expected_rq2_rows
assert rq2_annual_series["unit_id"].nunique() == int(rq2_config["expected_series_count"])
assert rq2_annual_series.groupby("unit_id", observed=True)["year"].nunique().eq(int(rq2_config["expected_n_per_series"])).all()
assert rq2.primary_trend_authority["realised_bandwidth"].nunique() == 1
assert rq2.primary_trend_authority["permutation_count"].eq(rq2_parameters.permutations).all()
assert len(rq2.primary_trend_authority[["raw_p", "bh_q"]].dropna()) == int(rq2_config["expected_series_count"])
assert tuple(rq2.primary_trend_authority.columns) == PRIMARY_TREND_COLUMNS
rq2_gate = "PASS"
display(rq2.realised_run_metadata)
display(pd.DataFrame([{"section": "02", "gate": rq2_gate}]))

# SECTION 03 — RQ3

RQ3 models cross-product observability rather than sensor accuracy. The descriptive population first classifies all paired district-months into four observation states. The inferential population then conditions on VIIRS-positive months and asks which pre-specified thermal-observation characteristics are associated with absence of corresponding MCD64A1 mapped burned surface. Exact factor-level outcome counts and complete-separation flags are audited before fitting; the frozen Firth-type PGEE, working independence and Morel–Bokossa–Neerchal corrected district-cluster sandwich are then used without feature selection.


### 03-01 — Paired correspondence, separation audit and PGEE authority

**Scientific purpose:** Execute the complete RQ3 descriptive and separation-resistant marginal model.

**Theoretical/methodological basis:** VIIRS thermal detections and MCD64A1 mapped burned surface are complementary observation processes. The fixed VIIRS-positive design uses configuration-defined log2 focal/adjustment predictors, categorical factor references, Firth-type PGEE, working independence, Morel–Bokossa–Neerchal covariance and standard-normal Wald inference. Separation is a property to report, not a reason to change the design. VIF is diagnostic only; predictive standardisation is defined at configured observed focal-predictor quantiles.

**Inputs:** Paired-overlap population and the frozen RQ3 configuration.

**Method:** Project the immutable RQ3 parameters and predictor-role metadata, build four-state correspondence, restrict to the configured VIIRS-positive population, construct the fixed design, audit separation, and run the qualified PGEE/MBN estimator.

**Expected output:** Complete finite RQ3 correspondence, coefficient and diagnostic authorities suitable for the configuration-registered model outputs.

**Interpretation limits:** Adjusted odds ratios are population-average associations with an observation-state outcome, not causal mechanisms or estimates of sensor error against ground truth.


In [ ]:
rq3_parameters = rq3_model_parameters(contracts.analysis)
rq3_predictor_roles = configured_predictor_metadata(contracts.analysis)
rq3 = build_rq3_tables(
    authorities,
    data_contract,
    contracts.analysis,
    contracts.methods,
    configuration_sha256=contracts.configuration_sha256,
)
rq3_parameter_audit = pd.DataFrame([{
    "estimator": rq3_parameters.estimator,
    "family": rq3_parameters.family,
    "link": rq3_parameters.link,
    "cluster_field": rq3_parameters.cluster_field,
    "working_correlation": rq3_parameters.working_correlation,
    "covariance": rq3_parameters.covariance,
    "reference_distribution": rq3_parameters.reference_distribution,
    "predictor_count": len(rq3_parameters.predictors),
    "factor_count": len(rq3_parameters.factors),
    "predictive_standardisation_method": rq3_parameters.predictive_standardisation_method,
    "predictive_quantiles": "|".join(str(x) for x in rq3_parameters.predictive_quantiles),
}])
display(rq3_parameter_audit)
display(rq3_predictor_roles)
display(rq3.four_state_summary_source)
display(rq3.continuous_model_summary_source)
display(rq3.standardised_probability_source)

### 03-02 — RQ3 acceptance gate

**Scientific purpose:** Verify exact governed populations, unchanged design dimension, convergence, finite inference and qualified reference parity.

**Theoretical/methodological basis:** Under separation, coefficient finiteness alone is insufficient; the estimator and corrected covariance must retain the independently qualified production authority.

**Inputs:** RQ3 model population, complete coefficient table and S5 qualification source.

**Method:** Compare realised observations/clusters with configured population authority, ensure every configured model term has finite inference, and require PASS numerical/reference statuses.

**Expected output:** RQ3 gate PASS.

**Interpretation limits:** The gate validates the frozen analytical path; it does not perform variable selection or choose a working-correlation structure.


In [ ]:
rq3_population_id = contracts.analysis.research_questions["rq3"]["model_population"]
rq3_expected = contracts.analysis.study["populations"][rq3_population_id]
rq3_fit = rq3.production_pgee.production.fit
assert len(rq3.model_population) == int(rq3_expected["rows"])
assert rq3_fit.n_clusters == int(rq3_expected["units"])
assert rq3_fit.design_dimension == len(rq3.gee_design.term_order)
assert rq3_fit.converged and rq3_fit.coefficients_finite and rq3_fit.covariance_finite
assert rq3.gee_design.reference_distribution == rq3_parameters.reference_distribution
assert not rq3.standardised_probability_source.empty
rq3_gate = "PASS"
display(pd.DataFrame([{"section": "03", "gate": rq3_gate, "design_columns": rq3_fit.design_dimension, "standardised_probability_rows": len(rq3.standardised_probability_source)}]))

# SECTION 04 — Secondary analysis

The secondary spatio-temporal analysis is analytically separate from RQ1–RQ3. Configuration permits exactly two SaTScan model identities: MCD64A1 retrospective space-time discrete Poisson with fixed burnable-area exposure, and VIIRS retrospective space-time permutation with **no exposure file**. Python deterministically generates and validates external-engine inputs. Cluster claims are only ingested after external outputs are provenance-linked to the exact parameter authority.


### 04-01 — SaTScan interface and external-result boundary

**Scientific purpose:** Generate the two configured scan interfaces and preserve the explicit external-execution state.

**Theoretical/methodological basis:** Scan-statistic search and Monte Carlo inference are external SaTScan responsibilities; the notebook must not substitute a Python approximation or interpret missing results as zero clusters.

**Inputs:** Validated district data/geometry and configured secondary-analysis authority.

**Method:** Call the package secondary-interface builder, which writes deterministic coordinates/cases, Poisson population only where authorised, exact parameter files and provenance registries. If validated external results are absent, retain `EXTERNAL_RESULTS_REQUIRED`.

**Expected output:** Two-model input/run registries and either validated cluster-result availability or explicit deferral.

**Interpretation limits:** `EXTERNAL_RESULTS_REQUIRED` means cluster-dependent T3/F6/S6 outputs are not yet scientific results; it is not equivalent to zero significant clusters. The notebook must never create synthetic cluster output.


In [ ]:
secondary_parameters = satscan_parameters(contracts.analysis)
secondary = build_secondary_input_run_isolated(
    paths=paths,
    run_id=run_ids["secondary"],
)
secondary_config = contracts.analysis.secondary_analysis
validate_secondary_completion_state(contracts.execution, secondary.execution_state)
configured_scan_ids = {item["scenario_id"] for item in secondary_config["scenarios"]}
assert secondary.scan_specification_registry["model_id"].nunique() == len(configured_scan_ids)
assert set(secondary.scan_specification_registry["model_id"]) == configured_scan_ids
secondary_gate = "PASS"
display(pd.DataFrame([asdict(secondary_parameters)]))
display(secondary.scan_specification_registry)
display(secondary.secondary_run_registry)
display(pd.DataFrame([{"section": "04", "gate": secondary_gate, "external_state": secondary.execution_state}]))


### 04-02 — Validate and integrate optional genuine SaTScan results

**Scientific purpose:** After 04-01 writes the deterministic SaTScan interfaces, the user executes SaTScan 10.3.3 externally. This cell then asks the package-owned integration API to validate any genuine result material and expose one governed external-result state to Section 90.

**Theoretical/methodological basis:** The generated `.prm` `ResultsFile` entry is the naming authority. For each governed model, the core family is the exact extensionless `ResultsFile`, `ResultsFile.col.txt` and `ResultsFile.gis.txt`. A duplicate `ResultsFile + ".txt"` is neither generated nor required, and an external `*.provenance.json` sidecar is neither generated nor required. Invalid or partial external material must fail closed rather than being interpreted as absence.

**Inputs:** The current run-local 04-01 `.prm`, `.cas`, `.geo` and, for MCD64A1 only, `.pop` authorities, plus any genuine externally generated core result-family files at the exact `ResultsFile` paths.

**Method:** Call the package-owned integration API. It verifies run-relative containment; current parameter/input hashes; the exact `ResultsFile` value; no VIIRS population authority; result freshness; successful completion; SaTScan 10.3.3; the governed model and study period; `.col`/`.gis` parsing; cluster/membership/location/centroid consistency; temporal limits; and STP non-Poisson relative-risk semantics. Python may record run-local validation metadata after success, but it does not fabricate external provenance or claim the local executable SHA.

**Expected output:** If neither governed family has genuine material, return `EXTERNAL_RESULTS_REQUIRED`. A single family or any incomplete/invalid family raises an error. Only two complete valid families return `RESULTS_VALIDATED`, after which Section 90 generates T3, F6 and S6 from run-local publication sources.

**Interpretation limits:** Result ingestion changes no SaTScan model, p-value, cluster selection, hierarchy or scientific interpretation. The notebook remains thin orchestration, does not execute SaTScan, does not create duplicate `.txt` reports or provenance sidecars, and never creates synthetic scientific cluster results.

In [ ]:
secondary_integration = validate_and_integrate_satscan_results_if_available(
    paths=paths,
    secondary_run_id=run_ids["secondary"],
)
assert secondary_integration.execution_state in {EXTERNAL_RESULTS_REQUIRED, RESULTS_VALIDATED}
secondary_gate = "PASS"
display(secondary_integration.as_status_frame())

# SECTION 90 — Integrated publication outputs

Publication outputs are resolved from the configuration-owned table and figure registries. The manuscript architecture comprises external Figure 1, generated F2–F6, T1–T3, S1–S6 and Figure S1; the generated estate is validated even when cluster-dependent products remain explicitly conditional on genuine external SaTScan results.


### 90-01 — Registered publication architecture and external boundary

**Scientific purpose:** Verify that configuration-registered publication identities remain resolvable after RQ2 production while preserving the external SaTScan boundary.

**Theoretical/methodological basis:** The primary RQ1–RQ3 authorities are complete, but cluster-dependent publication products cannot be treated as complete until external SaTScan outputs are validated. The notebook therefore does not fabricate or silently omit secondary inference.

**Inputs:** Validated RQ1–RQ3 authorities, secondary execution state and output/figure contracts.

**Method:** Resolve configured table/figure families. Materialise all internally available publication outputs. When 04-02 records `RESULTS_VALIDATED`, Section 90 consumes only the validated run-local SaTScan publication sources; otherwise T3/F6/S6 remain explicitly deferred.

**Expected output:** Configuration-owned publication identity registry and the complete publication registry with explicit availability states.

**Interpretation limits:** External-result deferral is not evidence of zero clusters and does not invalidate RQ2 production.


In [ ]:
PUBLICATION_RUN = build_publication_run_isolated(
    paths=paths,
    run_id=run_ids["publication"],
    secondary_run_id=run_ids["secondary"],
)
integration_validation = validate_canonical_integration(
    bundle=contracts,
    data_contract=data_contract,
    rq1=rq1,
    rq2=rq2,
    rq3=rq3,
    secondary=secondary_integration,
    publication_run=PUBLICATION_RUN,
)
publication_gate = integration_validation.status
display(PUBLICATION_RUN.table_registry)
display(PUBLICATION_RUN.figure_registry)
display(pd.DataFrame([integration_validation.summary]))
display(integration_validation.as_frame())

# SECTION 99 — Reproducibility closeout

The closeout records the exact inputs, software environment, five configuration hashes, method-reference state and output manifest for this execution. These records are provenance authorities only; they do not alter scientific tables or figures.


### 99-01 — Reproducibility manifest and method-reference state

**Scientific purpose:** Freeze the computational identities needed to reproduce the canonical run.

**Theoretical/methodological basis:** Scientific reproducibility requires both data/config identity and software/method qualification state. External-engine deferral must remain visible rather than silently discarded.

**Inputs:** Current configuration bundle, governed input paths, publication artefacts, method-authority contract and environment.

**Method:** Create a closeout run, export realised configuration, write run provenance, record method qualification states and write the integrated output manifest atomically.

**Expected output:** Reproducibility manifest, method-reference table and output manifest.

**Interpretation limits:** Package/version metadata document execution context; they are not empirical covariates or inferential quantities.


In [ ]:
closeout_dir = paths.create_run(run_ids["closeout"])
closeout_writer = OutputWriter(closeout_dir, contracts.output)
realised_configuration_path = export_realised_configuration(contracts, closeout_dir)
provenance = RunProvenance(
    run_id=run_ids["closeout"],
    project_root=paths.root,
    configuration_sha256=contracts.configuration_sha256,
    input_paths=list(paths.all_hashed_inputs),
    input_validation_status=input_validation_status,
    canonical_input_identity=canonical_input_identity,
)
provenance.finish()
provenance_path = provenance.write_manifest(closeout_dir / "99_closeout" / "M_REPRODUCIBILITY_MANIFEST.json")
method_reference_state = pd.DataFrame([
    {
        "method_id": method.method_id,
        "family": method.family,
        "status": method.status,
        "qualification_status": method.qualification_status,
        "outcome_tuning_allowed": method.outcome_tuning_allowed,
    }
    for method in contracts.methods.authorities
])
output_manifest = artefact_registry(PUBLICATION_RUN.run_dir)
closeout_writer.write_csv(
    "99_closeout/M_METHOD_REFERENCE_STATE.csv",
    method_reference_state.to_dict(orient="records"),
    columns=list(method_reference_state.columns),
    role="internal_authority",
)
closeout_writer.write_csv(
    "99_closeout/M_CANONICAL_INTEGRATION_VALIDATION.csv",
    integration_validation.as_frame().to_dict(orient="records"),
    columns=list(integration_validation.as_frame().columns),
    role="internal_authority",
)
closeout_writer.write_csv(
    "99_closeout/M_OUTPUT_MANIFEST.csv",
    output_manifest.to_dict(orient="records"),
    columns=list(output_manifest.columns),
    role="internal_authority",
)
display(method_reference_state)
display(output_manifest)

### 99-02 — Canonical execution state

**Scientific purpose:** Apply a final fail-closed state check across configuration, inputs, completed primary RQ components and the external secondary-analysis boundary.

**Theoretical/methodological basis:** RQ1, RQ2 and RQ3 production authorities can pass independently of live external SaTScan execution. The notebook must distinguish completed primary analysis from publication products that require external cluster results.

**Inputs:** Section gates, provenance files, configuration registry and publication state.

**Method:** Require scaffold, RQ1, RQ2, RQ3 and secondary-interface gates to pass; permit publication only as either complete or explicitly deferred for external results; preserve the exact current input hashes and separately record canonical-release byte identity.

**Expected output:** Primary-analysis execution PASS with external secondary/publication state recorded separately.

**Interpretation limits:** `EXTERNAL_RESULTS_REQUIRED` does not mean zero clusters and does not invalidate the completed RQ1–RQ3 primary analyses.


In [ ]:
assert scaffold_gate == rq1_gate == rq2_gate == rq3_gate == secondary_gate == "PASS"
assert publication_gate == "PASS"
assert integration_validation.status == "PASS"
assert input_validation_status == "PASS"
assert canonical_input_identity in {"MATCH", "DIFFERENT"}
assert realised_configuration_path.is_file()
assert provenance_path.is_file()
assert secondary_integration.execution_state in {EXTERNAL_RESULTS_REQUIRED, RESULTS_VALIDATED}
final_run_gate = "PASS"
final_run_summary = {
    "canonical_notebook": contracts.analysis.project["canonical_notebook"],
    "configuration_sha256": contracts.configuration_sha256,
    "input_validation_status": input_validation_status,
    "canonical_input_identity": canonical_input_identity,
    "scaffold_gate": scaffold_gate,
    "rq1_gate": rq1_gate,
    "rq2_gate": rq2_gate,
    "rq3_gate": rq3_gate,
    "secondary_gate": secondary_gate,
    "secondary_external_state": secondary_integration.execution_state,
    "publication_gate": publication_gate,
    "final_run_gate": final_run_gate,
}
closeout_writer.write_json(
    "99_closeout/M_CANONICAL_EXECUTION_GATE.json",
    final_run_summary,
    role="internal_authority",
)
display(pd.DataFrame([final_run_summary]))
interactive_finalisation = finalise_interactive_run(
    paths=paths,
    run_id=run_tag,
)
display(pd.DataFrame([interactive_finalisation.summary]))